# LLM APIs — Anthropic & OpenAI

Large Language Model APIs follow the same REST/JSON pattern as any other API, but have specific conventions worth understanding: the **messages format**, **roles**, **structured outputs**, and **tool use**.

We'll use the **Anthropic Claude API** as the primary example (it's the cleanest API design), with notes on OpenAI where the pattern differs.

---

## Setup

```bash
pip install anthropic          # Anthropic SDK
pip install openai             # OpenAI SDK (optional)
```

Store your API key as an environment variable — never hardcode it:

```bash
# .env file (add to .gitignore)
ANTHROPIC_API_KEY=sk-ant-...
```


In [ ]:
import os
import anthropic

# Reads ANTHROPIC_API_KEY from environment automatically
client = anthropic.Anthropic()

---

## 1. Basic Messages API

Every request sends a list of **messages** with alternating `user` / `assistant` roles. The model returns the next `assistant` message.

In [ ]:
message = client.messages.create(
    model='claude-sonnet-4-6',
    max_tokens=256,
    messages=[
        {'role': 'user', 'content': 'Explain overfitting in one sentence.'}
    ]
)

print(message.content[0].text)

### System prompt

The `system` parameter sets persistent instructions for the model's behaviour — persona, output format, constraints.

In [ ]:
message = client.messages.create(
    model='claude-sonnet-4-6',
    max_tokens=512,
    system='You are a data science tutor. Always give concrete Python examples. Be concise.',
    messages=[
        {'role': 'user', 'content': 'What is the difference between bagging and boosting?'}
    ]
)

print(message.content[0].text)

---

## 2. Multi-turn Conversations

Build up a conversation by appending to the messages list.

In [ ]:
messages = []

def chat(user_input):
    messages.append({'role': 'user', 'content': user_input})
    response = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=512,
        system='You are a concise data science tutor.',
        messages=messages
    )
    reply = response.content[0].text
    messages.append({'role': 'assistant', 'content': reply})
    return reply

print(chat('What is a confusion matrix?'))
print('---')
print(chat('Give me a Python example using sklearn.'))  # model remembers previous context

---

## 3. Structured Outputs (JSON Mode)

Force the model to return valid JSON by instructing it in the system prompt and asking it to wrap output in a JSON block. This is the standard pattern for **extraction** and **classification** tasks.

In [ ]:
import json

text = """
The random forest model achieved 94% accuracy on the test set but only 67% on the validation set,
suggesting significant overfitting. Training took 4 minutes on a MacBook M2.
"""

message = client.messages.create(
    model='claude-sonnet-4-6',
    max_tokens=256,
    system='Extract information and return ONLY valid JSON. No explanation.',
    messages=[{
        'role': 'user',
        'content': f'''Extract from this text:
- model_type
- train_accuracy (null if not mentioned)
- test_accuracy
- validation_accuracy
- overfitting (boolean)
- training_time_minutes

Text: {text}

Return as JSON.'''
    }]
)

result = json.loads(message.content[0].text)
print(json.dumps(result, indent=2))

---

## 4. Practical DS Patterns

### Text classification

In [ ]:
import pandas as pd

reviews = [
    'The model training was blazing fast and the results exceeded expectations.',
    'Terrible documentation, spent hours debugging a simple import error.',
    'Decent library but the API keeps changing between versions.',
]

def classify_sentiment(text):
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',  # use Haiku for cheap batch classification
        max_tokens=10,
        messages=[{
            'role': 'user',
            'content': f'Classify sentiment as POSITIVE, NEGATIVE, or NEUTRAL. Reply with only the label.\n\n{text}'
        }]
    )
    return msg.content[0].text.strip()

df = pd.DataFrame({'review': reviews})
df['sentiment'] = df['review'].apply(classify_sentiment)
df

### Summarisation

In [ ]:
long_text = """
Support Vector Machines (SVM) are supervised learning models that find the optimal hyperplane 
separating classes in a high-dimensional feature space. The key insight is maximising the margin — 
the distance between the hyperplane and the nearest data points (support vectors) from each class. 
SVMs can handle non-linear boundaries through the kernel trick, which implicitly maps data to a 
higher-dimensional space without explicit computation. Common kernels include RBF (Radial Basis 
Function), polynomial, and sigmoid. SVMs are effective in high-dimensional spaces and are memory 
efficient since only support vectors influence the decision boundary. However, they scale poorly 
to very large datasets and require careful kernel and hyperparameter selection.
"""

msg = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=80,
    messages=[{
        'role': 'user',
        'content': f'Summarise in one sentence (max 30 words):\n\n{long_text}'
    }]
)
print(msg.content[0].text)

---

## 5. Streaming

For long responses, stream tokens as they arrive rather than waiting for the full response.

In [ ]:
with client.messages.stream(
    model='claude-sonnet-4-6',
    max_tokens=300,
    messages=[{'role': 'user', 'content': 'Explain gradient descent step by step.'}]
) as stream:
    for text in stream.text_stream:
        print(text, end='', flush=True)

---

## Model Selection Guide (Anthropic, 2026)

| Model | ID | Use When |
|-------|----|----------|
| Claude Opus 4 | `claude-opus-4-7` | Complex reasoning, long context, best quality |
| Claude Sonnet 4 | `claude-sonnet-4-6` | Balanced — good default for most tasks |
| Claude Haiku 4 | `claude-haiku-4-5-20251001` | High-volume, simple tasks (classification, extraction) |

**Cost tip:** Use Haiku for batch processing, Sonnet for interactive use, Opus only when quality is critical.

---

## Summary

| Pattern | API call | Use case |
|---------|----------|----------|
| Single question | `messages.create(messages=[...])` | One-off queries |
| System prompt | `system='...'` | Persona, format constraints |
| Multi-turn | Append to `messages` list | Chatbots, iterative tasks |
| Structured output | JSON instruction in prompt | Extraction, classification |
| Streaming | `messages.stream(...)` | Long responses, UX |
| Cheap batch | Use Haiku model | Large-scale text processing |
